# 2. 설정과 데이터 읽어오기

In [1]:
!pip install pandas numpy matplotlib scikit-learn xgboost

## 1. 라이브러리 import

In [2]:
import pandas as pd # 표 형태 데이터를 다루는 라이브러리
import numpy as np # 수치 계산용 라이브러리 (RMSE 계산 등에 사용)
import matplotlib.pyplot as plt # 그래프 시각화 라이브러리

# train_test_split: 데이터를 학습용곽 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split
# 회귀 모델 평가에 사용할 지표들(MAE, MAE→RMSE, R2)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# ColumnTransformer: 컬럼 종류별로 다른 전처리를 적용하는 도구
from sklearn.compose import ColumnTransformer
# Pipeline: 전처리와 모델 학습을 하나의 흐름으로 묶는 도구
from sklearn.pipeline import Pipeline
# OneHotEncoder: 문자형(범주형) 값을 숫자 형태로 바꾸는 도구
from sklearn.preprocessing import OneHotEncoder
# SimpleImputer: 비어 있는 값(결측치)을 정해진 규칙으로 채우는 도구
from sklearn.impute import SimpleImputer

# XGBRegressor: 숫자 값을 예측하는 회귀용 XGBoost모델
from xgboost import XGBRegressor

In [ ]:
import pandas as pd# 표 형태 데이터를 다루는 라이브러리
import numpy as np# 수치 계산용 라이브러리 (RMSE 계산 등에 사용)
import matplotlib.pyplot as plt

# train_test_split: 데이터를 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split
# 회귀 모델 평가에 사용할 지표들(MAE, MAE→RMSE, R2)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# ColumnTransformer: 컬럼 종류별로 다른 전처리를 적용하는 도구
from sklearn.compose import ColumnTransformer
# Pipeline: 전처리와 모델 학습을 하나의 흐름으로 묶는 도구
from sklearn.pipeline import Pipeline
# OneHotEncoder: 문자형(범주형) 값을 숫자 형태로 바꾸는 도구
from sklearn.preprocessing import OneHotEncoder
# SimpleImputer: 비어 있는 값(결측치)을 정해진 규칙으로 채우는 도구
from sklearn.impute import SimpleImputer

# XGBRegressor: 숫자 값을 예측하는 회귀용 XGBoost모델
from xgboost import XGBRegressor

In [4]:
# 그래프의 모양을 설정 합니다.
plt.style.use("ggplot")

In [ ]:
import matplotlib.font_manager as fm
import warnings

# ---------------------------------------
# Windows 한글 폰트 설정
# ---------------------------------------
# matplotlib의 기본 글꼴은 Dejavu Sans인데, 이 글꼴을 한글을 제대로 표시하지 못합니다.
# 그래서 Windows에 기본으로 설치된 '맑은 고딕'을 그래프 글꼴로 저장합니다.
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스(-) 기호가 깨지는 문제를 방지합니다.
# 예: -10 같은 값이 □10 처럼 보이는 문제 예방
plt.rcParams['axes.unicode_minus'] = False

# 한글 글꼴 관련 경고가 너무 많이 출력되는 것을 줄입니다.
warnings.filterwarnings('ignore', category = UserWarning)

## 2. 데이터 로드

In [6]:
# 같은 폴더에 jeju_bus.csv 가 있다고 가정합니다.
# 파일이 다른 경로에 있다면 아래 경로를 수정합니다. 예: pd.read_csv('data/jeju_bus.csv')
df = pd.read_csv('jeju_bus.csv') # csv 파일을 읽어 DataFrame(df)으로 만듭니다.

# 원본 데이터는 보존하고, 모델링용으로는 복사본을 사용합니다.
# .copy()를 쓰면 df_model을 바꿔도 원본 df는 영향을 받지 않습니다.
df_model = df.copy()

## 3. 데이터 기본 확인

In [7]:
# 상위 5개 행을 확인합니다. 실제 값이 어떤 형태인지 눈으로 점검하는 단계입니다.
df.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,next_arrive_time
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,제대마을,33.457724,126.554014,24
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,제대아파트,33.458783,126.557353,36
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,제주대학교,33.459893,126.561624,40
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,제주여자중고등학교(아라방면),33.484860,126.542928,42
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,은남동,33.485822,126.490897,64


In [8]:
# 데이터의 (행 개수, 열 개수)를 확인합니다. 데이터 규모를 파악합니다.
df.shape

(210457, 14)

In [ ]:
# 실제 컬럼명을 확인합니다. 이후 코드(전처리, feature 목록 등)는 이 컬럼명을 기준으로 작성되어 있습니다.
# 만약 실제 컬럼명이 코드와 다르면, 이 출력을 기준으로 코드의 컬럼명을 맞춰야 합니다.
df.columns

Index(['id', 'date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude', 'next_arrive_time'],
      dtype='str')

In [10]:
# 컬럼별 결측치(비어 있는 값) 개수를 확인합니다. 값이 클수록 그 컬럼에 빈 값이 많다는 뜻입니다.
df.isnull().sum()

id                  0
date                0
route_id            0
vh_id               0
route_nm            0
now_latitude        0
now_longitude       0
now_station         0
now_arrive_time     0
distance            0
next_station        0
next_latitude       0
next_longitude      0
next_arrive_time    0
dtype: int64

# 4. 기본 정보 확인

## df.info()란 무엇인가요?

In [11]:
# 각 컬럼의 자료형(숫자/문자)과 결측치 여부를 한눈에 확인합니다.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 210457 entries, 0 to 210456
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                210457 non-null  int64  
 1   date              210457 non-null  str    
 2   route_id          210457 non-null  int64  
 3   vh_id             210457 non-null  int64  
 4   route_nm          210457 non-null  str    
 5   now_latitude      210457 non-null  float64
 6   now_longitude     210457 non-null  float64
 7   now_station       210457 non-null  str    
 8   now_arrive_time   210457 non-null  str    
 9   distance          210457 non-null  float64
 10  next_station      210457 non-null  str    
 11  next_latitude     210457 non-null  float64
 12  next_longitude    210457 non-null  float64
 13  next_arrive_time  210457 non-null  int64  
dtypes: float64(5), int64(4), str(5)
memory usage: 22.5 MB


# 5. 데이터 확인하기

In [12]:
df.describe()

,id,route_id,vh_id,now_latitude,now_longitude,distance,next_latitude,next_longitude,next_arrive_time
count,210457.000000,2.104570e+05,2.104570e+05,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000
mean,105228.000000,4.052491e+08,7.988694e+06,33.434528,126.603451,490.256100,33.434711,126.603687,85.380824
std,60753.847139,9.132404e+04,6.774077e+03,0.102350,0.123961,520.563932,0.102224,0.123838,85.051170
min,0.000000,4.051360e+08,7.983000e+06,33.244382,126.473300,97.000000,33.244382,126.473300,6.000000
25%,52614.000000,4.051365e+08,7.983093e+06,33.325283,126.523900,291.000000,33.325283,126.524550,44.000000
50%,105228.000000,4.053201e+08,7.983431e+06,33.484667,126.551050,384.000000,33.484860,126.551050,66.000000
75%,157842.000000,4.053201e+08,7.997041e+06,33.500197,126.650322,542.000000,33.500228,126.650322,102.000000
max,210456.000000,4.053281e+08,7.997124e+06,33.556167,126.935188,7461.000000,33.556167,126.935188,2996.000000
